# Spark version checker

Check Spark versions between this notebook pyspark and spark cluster
Check python versions between this notebook and spark cluster
Check java versions between this book and spark cluster

https://www.oracle.com/java/technologies/javase/jdk17-0-13-later-archive-downloads.html



In [1]:
import subprocess
import datetime
import os
import json
import re

spark_master = 'spark-master'
spark_workers = ['spark-worker-1', 'spark-worker-2']
python_package_filter = ['pyspark', 'delta-spark']
tfds_root = '/opt/tfds/data'
os.environ['SPARK_CONF_DIR'] = '/opt/tfds/spark/conf'

def extract_version(text: str, marker: str) -> str:
    # Find where the marker first appears
    marker_index = text.find(marker)
    if marker_index == -1:
        return None  # marker not found

    # Slice the text to search only after the marker
    after_marker = text[marker_index + len(marker):]

    # Search for the first version number after the marker
    match = re.search(r'\d+\.\d+\.\d+', after_marker)
    if match:
        return match.group(0)
    return None


def run_command(command, container_name=None):
    """Executes a command inside a Docker container and returns the output."""
    if container_name and container_name!='local':
        command = f"docker exec {container_name} {command}"
    try:
        result = subprocess.run(
            command,
            shell=True,                  # Allows passing the command as a single string
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,    # Redirect stderr to stdout
            text=True,
            check=True
        )
        return result.stdout.strip()
    except subprocess.CalledProcessError as e:
        return f"Error: {e.output.strip()}"


def get_spark_versions(container_name=None)->dict:
    result = {}
    result['spark_host'] = run_command(container_name=container_name, command="hostname")
    output = run_command(container_name=container_name, command="spark-shell --version")
    result['spark'] = extract_version(text=output, marker='version')
    result['scala'] = extract_version(text=output, marker='Scala')
    lines = output.split('\n')
    for l in lines:
        marker_index = l.find('OpenJDK')
        if marker_index != -1:
            result['OpenJDK'] = l[marker_index:]
    java = extract_version(text=output, marker='OpenJDK')
    if not java:
        output = run_command(container_name=container_name, command="java --version")
        java = extract_version(text=output, marker='java')
    result['java'] = java
    return result



def get_python_versions(container_name=None)->dict:
    result = {}
    result['python_host'] = run_command(container_name=container_name, command="hostname")
    output = run_command(container_name=container_name, command='python3 --version')
    result['python'] = extract_version(text=output, marker = 'Python')

    output = run_command(container_name=container_name, command='pip freeze --no-cache-dir')
    packages = {}
    for package in output.split('\n'):
        if not '==' in package:
            continue
        parts = package.split('==')
        packages[parts[0]] = parts[1]
    result['python_packages'] = packages
    return result


def get_all_versions()->list:
    version_dicts = []
    for container in ['local', spark_master] + spark_workers:

        version_dict = get_spark_versions(container_name=container)
        for key, value in get_python_versions(container_name=container).items():
            version_dict[key] = value
        version_dicts.append(version_dict)
    return version_dicts

def compare_versions(dict_list:list):
    keys = list(dict_list[0].keys())
    diff_dict = {}
    same_dict = {}
    for key in keys:
        if key in ('python_packages', 'spark_host', 'python_host'):
            continue
        host_value = {}
        for d in dict_list:
            host = d['spark_host']
            host_value[host] = d[key]
        if len(set(host_value.values())) != 1:
            diff_dict[key] = host_value
        else:
            same_dict[key] = dict_list[0][key]

    return diff_dict, same_dict

def compare_packages(dict_list:list, python_packages):
    keys = list(dict_list[0].keys())
    diff_dict = {}
    same_dict = {}

    for key in python_packages:

        host_value = {}
        for d in dict_list:
            host = d['spark_host']
            pkg_dict = d['python_packages']
            for package, version in d['python_packages'].items():
                if package==key:
                    host_value[host] = version
                    break
            else:
                # make sure the value is different in case none of them have the package
                host_value[host] = f'{host} does not have {key}'

        if len(set(host_value.values())) != 1:
            diff_dict[key] = host_value
        else:
            # we know all of the have it...and in the same version
            same_dict[key] = dict_list[0]['python_packages'][key]

    return diff_dict, same_dict

def check_file_system(data_root):
    # put a file on the local file system and check for it in all places
    folder_name = os.path.join(data_root, 'spark')
    file_name = os.path.join(folder_name, 'testfile_' + str(datetime.datetime.timestamp(datetime.datetime.now(tz=datetime.timezone.utc))))
    result = {}
    try:
        with open(file_name, 'w') as f:
            f.write('this is a testfile from spark_check.ipynb, it can be deleted')
            for host in ['local', spark_master] + spark_workers:
                result[host] = run_command(container_name=host, command=f'ls {file_name}')
            test = set(result.values())
            if len(test)!=1 or list(test)[0] != file_name:
                print("!!! Mounts are misconfigured, the test file was not found in some loctions")
            else:
                print("Mounts are all good, test file found in all locations")
            print(json.dumps(result, indent=4))
    except Exception as x:
        raise
    finally:
        os.remove(file_name)
    return result

import os
import requests
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from delta import configure_spark_with_delta_pip

os.environ['SPARK_CONF_DIR'] = '/opt/tfds/spark/conf'

def get_config(config_name):

    config_server_url = os.environ.get("TFDS_CONFIG_URL")
    if config_server_url is None:
        config_server_url = "http://tfds-config:8005/api/configs"

    config_url = config_server_url + "/" + config_name

    print(f"retrieving {config_name} config from {config_url}")
    response = requests.get(config_url)
    response.raise_for_status()
    if response.json() is None:
        raise ValueError(f"Config '{config_name}' not found. config server response: {response.text}")
    cfg = response.json().get("config")
    if cfg is None:
        raise ValueError(f"Config '{config_name}' does not have a 'config' key. Config server response: {response.text}")

    if config_name=='s3' and "TFDS_S3_URL" in os.environ.keys():
        cfg["url"] = os.environ["TFDS_S3_URL"]
    if config_name=='spark' and "TFDS_SPARK_MASTER_URL" in os.environ.keys():
        cfg["master_url"] = os.environ["TFDS_SPARK_MASTER_URL"]
    return cfg

def get_spark_session(use_local=False):
    s3_cfg = get_config("s3")

    conf = (
        pyspark.conf.SparkConf()
        .set("spark.hadoop.fs.s3a.access.key", s3_cfg["access_key"])
        .set("spark.hadoop.fs.s3a.secret.key", s3_cfg["secret_key"])
    )
    if use_local:
        conf.setMaster("local[*]")
    builder = pyspark.sql.SparkSession.builder.config(conf=conf)
    # spark_session = configure_spark_with_delta_pip(builder).getOrCreate()
    spark_session = builder.getOrCreate()
    return spark_session

def show_cfg(spark_session):
    cfg = spark_session.sparkContext.getConf().getAll()
    for key, value in cfg:
        if key in (
            'spark.submit.pyFiles',
            'spark.driver.extraJavaOptions',
            'park.app.initial.jar.urls',
            'spark.files',
            'spark.repl.local.jars',
            'spark.app.initial.file.urls'
            'spark.executor.extraJavaOption',
            'spark.app.initial.jar.urls'
            'spark.app.initial.file.urls'
            ):
            print(key)
            for l in value.split(','):
                print('    ' + str(l))
        else:
            print(f'{key} = {value}')

def print_spark_info(sc:SparkSession):
    cfg:pyspark.SparkConf = sc.sparkContext.getConf()
    print(f'==== spark app: {cfg.get("spark.app.name")} ====')
    print(f'Spark master: {cfg.get("spark.master")}')
    print(f'Delta lake location: {cfg.get("spark.sql.warehouse.dir")}')
    print(f'S3 endpoint: {cfg.get("spark.hadoop.fs.s3a.endpoint")}')

    dbs = sc.catalog.listDatabases()
    print("Databases:")
    for db in dbs:
        print(db.name)
        tables = sc.catalog.listTables(db.name)
        for tbl in tables:
            print(f'    {tbl.name}')

import subprocess



def list_jars_in_local_folder(folder_path):
    """
    List all JAR files in a specified local folder.

    :param folder_path: Path to the local folder.
    :return: List of JAR file names.
    """
    try:
        jars = [f for f in os.listdir(folder_path) if f.endswith(".jar")]
        return jars
    except FileNotFoundError:
        print(f"Error: Folder '{folder_path}' not found.")
        return []

def list_jars_in_docker(container_name, jar_path):
    """
    List all JAR files in a specified path inside a Docker container.

    :param container_name: Name of the Docker container.
    :param jar_path: Path inside the container to search for JAR files.
    :return: List of JAR file names.
    """
    try:
        # Construct the command to list JAR files in the specified path
        command = f"docker exec {container_name} ls {jar_path} | grep '.jar'"
        result = subprocess.run(
            command,
            shell=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            check=True
        )
        # Split the output into a list of JAR file names
        jars = result.stdout.strip().split("\n")
        return jars
    except subprocess.CalledProcessError as e:
        print(f"Error listing JARs in container '{container_name}': {e.stderr.strip()}")
        return []


def get_spark_jars(use_local):
    """
    Start a Spark session, retrieve the list of JARs, and stop the session.

    """
    spark = get_spark_session(use_local=use_local)
    jars = spark.sparkContext.getConf().get("spark.jars", "").split(",")


    spark.stop()
    # Combine all JARs into a single list
    return jars


def parse_jar_name_and_version(jar_path):
    """
    Extract the main name and version from a JAR file path.
    :param jar_path: Full path to the JAR file.
    :return: Tuple (name, version) or (name, None) if version is not found.
    """
    jar_name = jar_path.split("/")[-1]  # Extract the file name
    parts = jar_name.split("-")
    if len(parts) > 1 and parts[-1].endswith(".jar"):
        # slf4j-reload4j-1.7.36.jar
        version = parts[-1][:-4]  # Extract version (without .jar)
        name = "-".join(parts[:-1])
        return {'base_name':name, 'version':version, 'jar_name': jar_name}
    return {'base_name':jar_name, 'version': '', 'jar_name': jar_name}

def inspect_one(env_name, base_name, jars, other_jars):
    res = []

    if len(jars) == 0:
        jar_name = ','.join([x['jar_name'] for x in jars])
        res.append({
            'status': f'missing in {env_name}',
            'base_name': base_name,
            'jar_name': jar_name,
        })
    if len(jars) > 1:
        for l in jars:
            res.append({
                'status': f'multiple versions in {env_name}',
                'base_name': base_name,
                'jar_name': l['jar_name'],
            })
    return res

def inspect_jar(base_name, local_parsed, docker_parsed):

    locals = [x for x in local_parsed if x['base_name'] == base_name]
    dockers = [x for x in docker_parsed if x['base_name'] == base_name]

    result = {
        'base_name': base_name,
    }

    result['docker'] = inspect_one('docker', base_name, dockers, locals)
    result['local'] = inspect_one('local', base_name, locals, dockers)

    res = []
    for l in locals:
        result['jar_name'] = l['jar_name']
        for d in dockers:
            if l['version'] != d['version']:
                result['status'] = 'mismatch'
                res.append({
                    'status': 'version mismatch',
                    'base_name': base_name,
                    'local_version': l['jar_name'],
                    'docker_version': d['jar_name'],
                })
    result['comparison'] = res
    if len(result['docker'])>0 or len(result['local'])>0 or len(result['comparison']) > 0:
        result['status'] = 'ERROR'
    else:
        result['status'] = 'OK'

    return result

def compare_jars(local_jars, cluster_jars):
    """
    Compare the JARs used in local and cluster Spark sessions.
    :param local_jars: List of JARs from the local Spark session.
    :param cluster_jars: List of JARs from the cluster Spark session.
    :return: List of differences in versions.
    """
    local_parsed = [parse_jar_name_and_version(jar) for jar in local_jars]
    docker_parsed = [parse_jar_name_and_version(jar) for jar in cluster_jars]

    result = []
    if len(local_parsed) == 0:
        result.append({
            'status': 'no jars in local',
        })
    if len(docker_parsed) == 0:
        result.append({
            'status': 'no jars in docker',
        })
    if len(docker_parsed) != len(local_parsed):
        result.append({
            'status': 'diff in jars count',
            'local': len(local_parsed),
            'docker': len(docker_parsed),
        })
    base_names = set([x['base_name'] for x in local_parsed]).union(set([x['base_name'] for x in docker_parsed]))
    for base_name in base_names:
        result.append(inspect_jar(base_name, local_parsed, docker_parsed))
    return result


### Checking versions

In [ ]:
versions = get_all_versions()
dv,sv = compare_versions(versions)
dp,sp = compare_packages(versions, python_package_filter)
import json
if len(dv.keys()) == 0:
    print('Versions are all good, same in all locations:')
    print(json.dumps(sv, indent = 4))
else:
    print("!!! Versions mismatch:")
    print('diff:')
    print(json.dumps(dv, indent = 4))
    print('same:')
    print(json.dumps(sv, indent = 4))

if len(dp.keys()) == 0:
    print('Packages are all good, same in all locations:')
    print(json.dumps(sp, indent = 4))
else:
    print("!!! Package mismatch:")
    print('diff:')
    print(json.dumps(dp, indent = 4))
    print('same:')
    print(json.dumps(sp, indent = 4))
out = check_file_system(tfds_root)


## Checking Spark config and connectivity

In [ ]:
spark = get_spark_session(use_local = False)
# spark.sparkContext.setLogLevel("DEBUG")
spark.catalog.listDatabases()
print_spark_info(spark)
spark.stop()


## Check jar versions, local-master

In [ ]:
import sys
import os

local_jars = get_spark_jars(True)
docker_jars = get_spark_jars(False)

print('---- jars ----')
compare_jars(local_jars, docker_jars)

local_extra_jars = list_jars_in_local_folder('/opt/tfds/spark/jars')
docker_extra_jars = list_jars_in_docker(container_name=spark_master, jar_path='/opt/tfds/spark/jars')
if len(local_extra_jars) == 0 or len(docker_extra_jars) != len(local_extra_jars):
    print("!!! Extra jars are not the same in local and docker")
    print(f'local: {len(local_extra_jars)}')
    print(f'docker: {len(docker_extra_jars)}')
print('---- extra jars ----')
compare_jars(local_extra_jars, docker_extra_jars)

local_jar_path = os.path.join(sys.prefix, 'lib/python3.8/site-packages/pyspark/jars')
local_spark_jars = list_jars_in_local_folder(local_jar_path)

docker_jar_path = '/opt/spark/jars'
docker_spark_jars = list_jars_in_docker(container_name=spark_master, jar_path=docker_jar_path)
if len(local_spark_jars) == 0 or len(docker_spark_jars) != len(local_spark_jars):
    print("!!! Spark jars are not the same in local and docker")
    print(f'local: {len(local_spark_jars)}')
    print(f'docker: {len(docker_spark_jars)}')
print('---- spark jars ----')
compare_jars(local_spark_jars, docker_spark_jars)
# apparently spark does not find these the pyspark ones, and pyspark is not executing our jobs...
# installing guava using pip works locally through pyspark, but not in docker
# docker_pyspark_jars = list_jars_in_docker(container_name=spark_master, jar_path='/usr/local/lib/python3.8/dist-packages/pyspark/jars/')

total_local_jars = list(set(local_spark_jars + local_extra_jars + local_jars))
total_docker_jars = list(set(docker_spark_jars + docker_extra_jars + docker_jars))
print('---- total jars ----')
res = compare_jars(total_local_jars, total_docker_jars)
if len(total_local_jars) == 0 or len(total_local_jars) != len(total_docker_jars):
    print("!!! Spark jars are not the same in local and docker")
    print(f'local: {len(total_local_jars)}')
    print(f'docker: {len(total_docker_jars)}')
for r in res:
    if r['status'] != 'OK':
        print (json.dumps(r, indent=4))
        continue
else:
    print('all ok')
# print (json.dumps(res, indent=4))


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import os
os.environ['SPARK_CONF_DIR'] = '/opt/tfds/spark/conf'

# Create Spark session

spark = (
    SparkSession.builder
    .appName("EvenNumbers")
    .master("local[2]")
    .getOrCreate()
)

data = spark.range(100)
# Filter even numbers
even_numbers = data.filter(col("id") % 2 == 0)
even_numbers.show(5)

spark.stop()

In [4]:
from pyspark.sql.functions import col
import os

spark = get_spark_session(use_local = False)
print_spark_info(spark)

data = spark.range(100)
# Filter even numbers
even_numbers = data.filter(col("id") % 2 == 0)
even_numbers.show(5)

spark.stop()

retrieving s3 config from http://tfds-config:8005/api/configs/s3
==== spark app: WhenDoIGetToSparkMyWay ====
Spark master: spark://spark-master:7077
Delta lake location: s3a://dwh/warehouse/
S3 endpoint: http://s3-minio:9000


Databases:
bronze


25/05/13 11:11:42 WARN ObjectStore: Failed to get database global_temp, returning NoSuchObjectException


    wikipedia_page_reads
default
silver
    wikipedia_page_ranks_100
+---+
| id|
+---+
|  0|
|  2|
|  4|
|  6|
|  8|
+---+
only showing top 5 rows

